In [ ]:
import anndata as ad
import squidpy as sq
import pandas as pd
import scanpy as sc
import numpy as np
import os
import geopandas as gpd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
tma6 = sc.read("/homevol/kk/analysis/OVA_TMA/TMA6_notebook/adjustments_mis_assigned_transcripts/supervised_annotation/v3/tma6_xen_mist_coarse_decoupler_filt_fine_sp_coord_adj.h5ad")

In [ ]:
adata = tma6.copy()

In [ ]:
## first to estimate which minimum redius values to use to count at least 3 unique neighbouring cells

In [ ]:
cores = {
    core: adata[adata.obs["arrayID"] == core].copy()
    for core in adata.obs["arrayID"].unique()
}

nbrDict = {}

radii_um = [3, 5, 10, 25, 50, 75, 100, 250, 500]

for r_um in radii_um:
    nbrDict[r_um] = []

    for core, ad_sub in cores.items():
        key = f"spatial_{r_um}um_{core}"

        sq.gr.spatial_neighbors(
            ad_sub,
            radius=r_um,          # <-- use directly
            coord_type="generic",
            spatial_key="spatial",
            key_added=key,
        )

        adj = ad_sub.obsp[f"{key}_connectivities"]
        counts = np.diff(adj.indptr)

        nbrDict[r_um].extend(counts)

In [ ]:
nbrDF = pd.DataFrame(nbrDict)
print(nbrDF)
print(nbrDF.median())

In [ ]:
nbrDF["arrayID"] = adata.obs["arrayID"].values
nbrDF.groupby("arrayID").median()


In [ ]:
## Calculation of non-tumour neighbouring cell-types around tumour cells

In [ ]:
# -----------------------------------------
# Configuration
# -----------------------------------------
radii_um = [3, 5, 10, 25, 50, 75, 100, 250, 500]
n_perm = 1000

celltype_col = "decoupler_fine"
tumour_types = ["Tumour", "Proliferating"]

rng = np.random.default_rng(42)

# -----------------------------------------
# Per-core AnnData objects
# -----------------------------------------
cores = {
    core: adata[adata.obs["arrayID"] == core].copy()
    for core in adata.obs["arrayID"].unique()
}

# Global cell order
cellOrder = adata.obs[celltype_col].value_counts().index.tolist()
non_tumour_types = [ct for ct in cellOrder if ct not in tumour_types]

results = {}

type_index = {t: i for i, t in enumerate(non_tumour_types)}

# -----------------------------------------
# Main loop
# -----------------------------------------
for r_um in radii_um:

    results[r_um] = {}

    for core, ad_sub in cores.items():

        # -----------------------------------------
        # Build spatial graph
        # -----------------------------------------
        key = f"spatial_{r_um}um_{core}"

        sq.gr.spatial_neighbors(
            ad_sub,
            spatial_key="spatial",
            coord_type="generic",
            radius=r_um,      
            key_added=key,
        )

        A = ad_sub.obsp[f"{key}_connectivities"].tocsr()
        ct = ad_sub.obs[celltype_col].values

        rows, cols = A.nonzero()

        # remove self edges
        mask_self = rows != cols
        rows = rows[mask_self]
        cols = cols[mask_self]

        src = ct[rows]
        tgt = ct[cols]

        mask = np.isin(src, tumour_types) & (~np.isin(tgt, tumour_types))

        tumour_arr = src[mask]
        neigh_arr = tgt[mask]

        # -----------------------------------------
        # Empty case
        # -----------------------------------------
        if len(tumour_arr) == 0:

            results[r_um][core] = {
                "real_counts": pd.DataFrame(0, index=non_tumour_types, columns=tumour_types),
                "coarse_counts": pd.Series(0, index=non_tumour_types),
            }
            continue

        # -----------------------------------------
        # Convert neighbour labels → integer ids
        # -----------------------------------------
        neigh_ids = np.array([type_index[n] for n in neigh_arr])
        

        
        coarse_counts_arr = np.bincount(
            neigh_ids,
            minlength=len(non_tumour_types)
        )

        coarse_counts = pd.Series(
            coarse_counts_arr,
            index=non_tumour_types
        ).astype(int)

        # -----------------------------------------
        # Store results
        # -----------------------------------------
        results[r_um][core] = {
            "coarse_counts": coarse_counts,
        }



In [ ]:
long_coarse = []

for r_um, core_dict in results.items():
    for core, block in core_dict.items():

        coarse_counts = block["coarse_counts"] 

        # Skip empty cores
        if coarse_counts is None:
            continue

        # Extract PatientID
        patient_id = adata.obs.loc[adata.obs["arrayID"] == core, "PatientID"].unique()
        patient_id = patient_id[0] if len(patient_id) == 1 else None

        # Extract Diagnosis
        diagnosis = adata.obs.loc[adata.obs["arrayID"] == core, "Diagnosis"].unique()
        diagnosis = diagnosis[0] if len(diagnosis) == 1 else None

        # Build long-format table
        df_long = (
            coarse_counts
            .reset_index()
            .rename(columns={"index": "neighbour_type", 0: "count"})
        )

        df_long["tumour_type"] = "Coarse"
        df_long["radius_um"] = r_um
        df_long["arrayID"] = core
        df_long["PatientID"] = patient_id
        df_long["Diagnosis"] = diagnosis

        long_coarse.append(df_long)

# Final long-format table
long_coarse = pd.concat(long_coarse, ignore_index=True)
long_coarse

In [ ]:
long_coarse_tum = long_coarse[long_coarse["Diagnosis"] == "Tumour"].copy()

tmp = (
    long_coarse_tum
    .groupby(["PatientID", "Diagnosis", "radius_um", "tumour_type", "neighbour_type"])["count"]
    .sum()
    .reset_index()
)

# Normalize within each (PatientID, Diagnosis, radius_um, tumour_type)
tmp["fraction"] = (
    tmp["count"] /
    tmp.groupby(["PatientID", "Diagnosis", "radius_um", "tumour_type"])["count"].transform("sum")
)

patient_frac_coarse = tmp

print(long_coarse_tum["neighbour_type"].nunique())

patient_frac_coarse.head(20)

## calculate all cell-cell touches

In [ ]:
## for decouplr broad labels

In [ ]:
# -----------------------------------------
# Configuration
# -----------------------------------------
radii_um = [3, 5, 10, 25, 50, 75, 100, 250, 500]
celltype_col = "decoupler_fine"

# -----------------------------------------
# Per-core AnnData objects
# -----------------------------------------
cores = {
    core: adata[adata.obs["arrayID"] == core].copy()
    for core in adata.obs["arrayID"].unique()
}

# Global cell-type order
cellOrder = adata.obs[celltype_col].value_counts().index.tolist()

# Storage
all_results = {}

# -----------------------------------------
# Main loop
# -----------------------------------------
for r_um in radii_um:

    r_px = r_um   # <-- use microns directly
    all_results[r_um] = {}

    for core, ad_sub in cores.items():

        key = f"spatial_{r_um}um_{core}"

        sq.gr.spatial_neighbors(
            ad_sub,
            spatial_key="spatial",
            coord_type="generic",
            radius=r_px,
            key_added=key,
        )

        A = ad_sub.obsp[f"{key}_connectivities"].tocsr()
        ct = ad_sub.obs[celltype_col].values

        rows, cols = A.nonzero()
        mask_self = rows != cols
        rows = rows[mask_self]
        cols = cols[mask_self]

        src = ct[rows]
        tgt = ct[cols]

        df = pd.DataFrame({"src": src, "tgt": tgt})

        touch_counts = (
            df.groupby(["src", "tgt"], observed=True)
              .size()
              .unstack(fill_value=0)
              .reindex(index=cellOrder, columns=cellOrder, fill_value=0)
        )

        all_results[r_um][core] = {
            "counts": touch_counts
        }


In [ ]:
long_all = []

for r_um, core_dict in all_results.items():
    for core, block in core_dict.items():

        counts = block["counts"]

        if counts is None or counts.empty:
            continue

        # Ensure index has a name for melt()
        counts = counts.copy()
        counts.index.name = "src_type"

        # Metadata
        patient_id = adata.obs.loc[adata.obs["arrayID"] == core, "PatientID"].unique()
        patient_id = patient_id[0] if len(patient_id) == 1 else None

        diagnosis = adata.obs.loc[adata.obs["arrayID"] == core, "Diagnosis"].unique()
        diagnosis = diagnosis[0] if len(diagnosis) == 1 else None

        # Melt
        df_long = (
            counts
            .reset_index()
            .melt(id_vars="src_type", var_name="tgt_type", value_name="count")
        )

        df_long["radius_um"] = r_um
        df_long["arrayID"] = core
        df_long["PatientID"] = patient_id
        df_long["Diagnosis"] = diagnosis

        long_all.append(df_long)

long_all = pd.concat(long_all, ignore_index=True)


long_all

In [ ]:
long_all_tum = long_all[long_all["Diagnosis"] == "Tumour"].copy()

long_all_tum = long_all_tum[~long_all_tum["tgt_type"].isin(bad_types)].copy()


print(long_all_tum["src_type"].nunique())
print(long_all_tum["tgt_type"].nunique())

long_all_tum

In [ ]:
long_all_tum = long_all[long_all["Diagnosis"] == "Tumour"].copy()

patient_sum = (
    long_all_tum
    .groupby(["PatientID", "radius_um", "src_type", "tgt_type"], observed=True)["count"]
    .sum()
    .reset_index()
)

patient_sum["fraction"] = (
    patient_sum["count"] /
    patient_sum.groupby(["PatientID", "radius_um", "src_type"], observed=True)["count"].transform("sum")
)

patient_sum["interaction"] = (
    patient_sum["src_type"] + "_to_" + patient_sum["tgt_type"]
)

patient_sum.head(50)